Swati , Jorge Angon, George Rodriguez

IS 170 lab 7

https://archive.ics.uci.edu/dataset/2/adult

In [ ]:
# ==========================================================
# Polynomial SVM - Adult Income Dataset
# This code loads the Adult dataset, cleans it,
# converts text columns into numbers, scales the data,
# trains a Polynomial SVM model, and evaluates the results.
# ==========================================================
# Import Libraries

import pandas as pd                                  # for working with tables/dataframes
from sklearn.model_selection import train_test_split # for splitting data into training and testing sets
from sklearn.preprocessing import StandardScaler     # for scaling numeric values
from sklearn.svm import SVC                          # Support Vector Classifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # evaluation metrics
from sklearn.preprocessing import PolynomialFeatures # for creating polynomial features

In [ ]:
# ---------------------------
# 2. Define column names
# ---------------------------
# The Adult dataset does not come with column names inside the file,
# so we assign them manually.
columns = [
    'age',
    'workclass',
    'fnlwgt',
    'education',
    'education_num',
    'marital_status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'capital_gain',
    'capital_loss',
    'hours_per_week',
    'native_country',
    'income'
]


In [ ]:
# ---------------------------
# 3. Load the dataset
# ---------------------------
# Read the file named "adult.data"
from google.colab import files

uploaded = files.upload()



Saving adult.data to adult.data


In [ ]:
import pandas as pd

df = pd.read_csv("adult.data", header=None)

In [ ]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
#ADD columns
# ==========================================================
# The dataset does not include column headers, so we define them manually

columns = ['age','workclass','fnlwgt','education','education_num',
           'marital_status','occupation','relationship','race','sex',
           'capital_gain','capital_loss','hours_per_week','native_country','income']

# Assign column names to dataframe
df.columns = columns

# Display first few rows
print("First 5 rows with column names:")
print(df.head())


First 5 rows with column names:
   age          workclass  fnlwgt   education  education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital_status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital_gain  capital_loss  hours_per_week  native_country  income  
0          2174             0       

In [ ]:
#  Clean missing data
# ==========================================================
# The dataset contains missing values represented as " ?"
# Replace them with NA (Not Available)

df = df.replace(' ?', pd.NA)

# Remove rows with missing values
df = df.dropna()

# Show updated shape
print("\nShape after removing missing values:", df.shape)



Shape after removing missing values: (30162, 15)


In [ ]:
# Convert categorical variables into numeric
# ==========================================================
# SVM cannot work with text data (like "Private", "Bachelors")
# So we convert categories into 0/1 columns using one-hot encoding

df = pd.get_dummies(df, drop_first=True)

# Display first few rows after encoding
print("\nFirst 5 rows after encoding:")
print(df.head())



First 5 rows after encoding:
   age  fnlwgt  education_num  capital_gain  capital_loss  hours_per_week  \
0   39   77516             13          2174             0              40   
1   50   83311             13             0             0              13   
2   38  215646              9             0             0              40   
3   53  234721              7             0             0              40   
4   28  338409             13             0             0              40   

   workclass_ Local-gov  workclass_ Private  workclass_ Self-emp-inc  \
0                 False               False                    False   
1                 False               False                    False   
2                 False                True                    False   
3                 False                True                    False   
4                 False                True                    False   

   workclass_ Self-emp-not-inc  ...  native_country_ Puerto-Rico  \
0     

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
print(df.columns)

Index(['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
       'hours_per_week', 'workclass_ Local-gov', 'workclass_ Private',
       'workclass_ Self-emp-inc', 'workclass_ Self-emp-not-inc',
       'workclass_ State-gov', 'workclass_ Without-pay', 'education_ 11th',
       'education_ 12th', 'education_ 1st-4th', 'education_ 5th-6th',
       'education_ 7th-8th', 'education_ 9th', 'education_ Assoc-acdm',
       'education_ Assoc-voc', 'education_ Bachelors', 'education_ Doctorate',
       'education_ HS-grad', 'education_ Masters', 'education_ Preschool',
       'education_ Prof-school', 'education_ Some-college',
       'marital_status_ Married-AF-spouse',
       'marital_status_ Married-civ-spouse',
       'marital_status_ Married-spouse-absent',
       'marital_status_ Never-married', 'marital_status_ Separated',
       'marital_status_ Widowed', 'occupation_ Armed-Forces',
       'occupation_ Craft-repair', 'occupation_ Exec-managerial',
       'occupation_ Farmi

In [ ]:
print([col for col in df.columns if 'income' in col])

['income_ >50K']


In [ ]:
# Split features (X) and target (y)
# ==========================================================
# X = all input variables
# y = output variable (income classification)
X = df.drop('income_ >50K', axis=1)
y = df['income_ >50K']

In [ ]:
# Split into training and testing sets
# ==========================================================
# 80% of data used for training, 20% for testing

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("\nTraining set size:", X_train.shape)
print("Testing set size:", X_test.shape)


Training set size: (24129, 96)
Testing set size: (6033, 96)


In [ ]:
# Scale the data
# ==========================================================
# SVM works better when features are on similar scale

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit scaler on training data and transform both sets
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Train Polynomial SVM model
# ==========================================================
# kernel='poly' → polynomial kernel
# degree=3 → cubic curve decision boundary

from sklearn.svm import SVC

model = SVC(kernel='poly', degree=3, C=1.0, gamma='scale')

# Train the model
model.fit(X_train, y_train)

SVC(kernel='poly')

In [ ]:
# Make predictions
# ==========================================================
# Predict income class for test data

y_pred = model.predict(X_test)

In [ ]:
#  Evaluate the model
# ==========================================================
# Accuracy = overall correct predictions
# Confusion Matrix = detailed prediction breakdown
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display, HTML

# Generate results
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

# Display in a styled box
display(HTML(f"""
<div style="border:3px solid #4CAF50; padding:15px; border-radius:10px; background-color:#f9fff9;">
    <h3 style="color:#2E7D32;">📊 Model Results</h3>

    <h4 style="color:#1565C0;">Confusion Matrix</h4>
    <pre style="background-color:#e3f2fd; padding:10px; border-radius:5px;">{cm}</pre>

    <h4 style="color:#6A1B9A;">Classification Report</h4>
    <pre style="background-color:#f3e5f5; padding:10px; border-radius:5px;">{report}</pre>
</div>
"""))



The dataset was preprocessed by eliminating missing values and using one-hot encoding to transform category variables into numeric representation. StandardScaler was then used to scale the data in order to enhance SVM performance. Non-linear connections between characteristics including age, education, and working hours were captured using a polynomial kernel. Accuracy, a confusion matrix, and a classification report were used to assess the model.